# Ring yields 3 — Fisher metric and validation

* Fisher metric of the cluster yields over (θ, E_AB) at fixed density (`analysis/fisher.py`), from ensemble scores `u = ⟨−β ∂U⟩` that include the fugacity response. The θ score omits the bond-indicator boundary term.
* Comparison with canonical MD trajectories (`analysis/validate_yields.py`) when they exist: mass action at the simulated density, and the finite-N recursion.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'base').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from base.jax_setup import configure_jax
configure_jax()
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
from base.paths import PARTITION_DIR, PLOTS_DIR, DATA_DIR, ensure_dir
FIG_DIR = ensure_dir(PLOTS_DIR / 'ring_yields')

In [ ]:
from analysis import yield_maps as ym
phi = 0.2
fine = ym.refine(ym.load_scan())
maps = ym.cached_maps(fine, phi)
g = maps.metric
# metric in (theta [rad], ln E_AB) coordinates: rescale the E_AB row/column by E_AB
E = maps.eabs[:, None]
g_ln = g.copy()
g_ln[..., 1, :] *= E[..., None]; g_ln[..., :, 1] *= E[..., None]
vol = np.sqrt(np.clip(np.linalg.det(g_ln), 0, None))
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for ax, Z, lab in zip(axes, [np.log10(g_ln[..., 0, 0]), np.log10(g_ln[..., 1, 1]), np.log10(vol)],
                      ['log10 g_θθ', 'log10 g_lnE lnE', 'log10 √det g']):
    m = ym._mesh(ax, maps, Z, cmap=ym.SEQUENTIAL)
    plt.colorbar(m, ax=ax, label=lab); ym._axes_style(ax, maps)
fig.suptitle(f'cluster-yield Fisher metric, φ={phi}')
fig.savefig(FIG_DIR / f'fisher_metric_phi{phi}.png', dpi=150); plt.show()

## MD comparison
Point `TRAJ` at canonical-MD trajectories at one design point (several seeds). The prediction uses the scanned point nearest in (θ, E_AB).

In [ ]:
from analysis.validate_yields import compare
from base.paths import DATA_DIR
TRAJ = sorted((DATA_DIR / 'nvt').glob('**/*.h5')) if (DATA_DIR / 'nvt').exists() else []
print(len(TRAJ), 'trajectories found')
if TRAJ:
    from base.trajectory import open_trajectory
    with open_trajectory(TRAJ[0]) as r:
        attrs = dict(r.attrs)
    scan = ym.load_scan()
    th = np.degrees(float(attrs['opening_angle']))
    i = int(np.argmin(np.abs(scan.eabs - float(attrs['E_AB']))))
    j = int(np.argmin(np.abs(scan.thetas_deg - th)))
    print(f'nearest scan point θ={scan.thetas_deg[j]}, E_AB={scan.eabs[i]}')
    cmp_ = compare(scan.table(i, j).finite(), TRAJ[:4])
    print(cmp_.table()); print('chi2 vs finite-N', round(cmp_.chi2(), 2))